# Experiment 4: Support Vector Machines (SVM) Kernel and Hyperparameter Tuning
This standalone notebook evaluates Support Vector Classification across diverse kernels (Linear, Polynomial, RBF, Sigmoid) on the Diabetes classification benchmark.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex4."""
    if os.path.basename(os.getcwd()) == 'Ex4':
        if rel_path.startswith('Ex4/'):
            return rel_path[len('Ex4/'):]
    return rel_path

import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex4/plots'), exist_ok=True)

In [2]:
def run_experiment_4(csv_path="Datasets/Diabetes_Dataset/diabetes.csv"):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 4: SUPPORT VECTOR MACHINES ===")
    print("="*60)
    
    path = resolve_path(csv_path)
    df = pd.read_csv(path)
    
    target_col = 'Outcome' if 'Outcome' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    
    kernels = ['linear', 'poly', 'rbf', 'sigmoid']
    results = {}
    
    for k in kernels:
        model = SVC(kernel=k, probability=True, random_state=42)
        model.fit(X_tr_sc, y_tr)
        y_pred = model.predict(X_te_sc)
        y_prob = model.predict_proba(X_te_sc)[:, 1]
        
        results[f'SVM ({k})'] = {
            'Accuracy': round(accuracy_score(y_te, y_pred) * 100, 2),
            'Precision': round(precision_score(y_te, y_pred, zero_division=0) * 100, 2),
            'Recall': round(recall_score(y_te, y_pred, zero_division=0) * 100, 2),
            'F1-Score': round(f1_score(y_te, y_pred, zero_division=0) * 100, 2),
            'ROC AUC': round(roc_auc_score(y_te, y_prob), 4)
        }
        
    print("\n=== EXPERIMENT 4 PIPELINE COMPLETE ===")
    return results

In [3]:
# Master Execution Cell
ex4_output = run_experiment_4()
display(pd.DataFrame(ex4_output).T.style.background_gradient(cmap='Purples', subset=['Accuracy', 'F1-Score']))

=== LAUNCHING EXPERIMENT 4: SUPPORT VECTOR MACHINES ===



=== EXPERIMENT 4 PIPELINE COMPLETE ===


,Accuracy,Precision,Recall,F1-Score,ROC AUC
SVM (linear),72.080000,62.220000,51.850000,56.570000,0.828100
SVM (poly),75.320000,73.530000,46.300000,56.820000,0.796100
SVM (rbf),75.320000,66.000000,61.110000,63.460000,0.792400
SVM (sigmoid),70.780000,60.000000,50.000000,54.550000,0.724400
